In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [4]:
df = pd.read_csv("/content/qoute_dataset.csv") #dataset of quotes

In [5]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [6]:
df.shape

(3038, 2)

In [7]:
quotes = df["quote"]
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


**Data Preprocessing**

(before applying lstm or anything as such , we nede to preprocess our data, and for that we needa use NLP techniques like vectorization , embeddings , tokenization etc)

In [8]:
quotes = quotes.str.lower()

In [9]:
#removing punctuations
import string
translator = str.maketrans('', '', string.punctuation) #for every punctuation replace it with empty space
quotes = quotes.apply(lambda x: x.translate(translator)) # "lambda x" -> every single quote in the dataset gets passed one by one



In [10]:
#tokenization (assign a number to each word)
from tensorflow.keras.preprocessing.text import Tokenizer

In [11]:
#the actual vocab size for our df can get pretty large sometimes(eg : 30lakh), but setting such a large number, we dont have enuf cpu nor gpu power for our model to compute that
vocab_size = 10000 #10k is a safe number which hopefully captures all the important words
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [12]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10] #8978 unique words and the top 10 most occuring words is printed

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [13]:
sequence = tokenizer.texts_to_sequences(quotes)

The quotes and their corresponding sequentially tokenized vectors are displayed

In [14]:
for i in range(3):
  print(quotes[i])


“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [15]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


For our input and output variable, our project is sentence completion, i.e, we are actively predicting the next word

eg: look at the first quote, if we write "the" [713] , it should predict "world" [62]

similarly if we write "the world" [713,62]. output : "as" [29]

In [16]:
X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    X.append(seq[:i]) #input sequence
    y.append(seq[i]) #output sequence

In [17]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [18]:
#padding
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding = "pre")

In [19]:
X_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]], dtype=int32)

In [20]:
y = np.array(y)

Our models like LSTM or GRU work with probabilities (0 and 1), so we need to one hot encode our output variable (y)
suppose y stores a tokenized word [121]
its one hot encoded to the vocab size, in it the position where the word is stored is encoded to 1 , the rest is 0, basically 9.9k 0's and only a single 1

[121] -> [0,0,0......1,.....0,0,0]

In [21]:
#one hot encoding
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [22]:
y_one_hot.shape

(85271, 10000)

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SimpleRNN

In [24]:
embedding_units = 50
rnn_units = 128

**Note** : The code application of RNN LSTM and GRU is mostly similar, only the internal working is different

In [25]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_units, input_length=max_len) #this layer creates the embeddings
)

rnn_model.add(
    SimpleRNN(units=rnn_units) #the embeddings come under this simple RNN layer (128 units)
)
rnn_model.add(
    Dense(units=vocab_size, activation="softmax") #output layer
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [26]:
rnn_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [27]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [28]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_units, input_length=max_len)
)
lstm_model.add(
    LSTM(units=rnn_units)
)
lstm_model.add(
    Dense(units=vocab_size, activation="softmax")
)

In [29]:
lstm_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [30]:
epochs = 10
batch_size = 128
history_rnn = rnn_model.fit(X_padded, y_one_hot, epochs=epochs, batch_size=batch_size, validation_split=0.1)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 50s 75ms/step - accuracy: 0.0440 - loss: 6.7254 - val_accuracy: 0.0536 - val_loss: 6.5511
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.0427 - loss: 7.6090 - val_accuracy: 0.0439 - val_loss: 7.7077
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.0655 - loss: 6.4966 - val_accuracy: 0.0639 - val_loss: 6.5386
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.0752 - loss: 6.2066 - val_accuracy: 0.0782 - val_loss: 6.6363
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 66ms/step - accuracy: 0.0805 - loss: 6.2618 - val_accuracy: 0.0789 - val_loss: 6.6791
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.0902 - loss: 6.0380 - val_accuracy: 0.0867 - val_loss: 6.5713
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 66ms/step - accuracy: 0.0993 - loss: 5.8923 - val_accuracy: 0.0886 - val_loss: 6.5377
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 66ms/step - accuracy: 0.1058 - loss: 5.7693 - 

In [31]:
from tensorflow.keras.callbacks import EarlyStopping
epochs_lstm = 100
es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history_lstm = lstm_model.fit(X_padded, y_one_hot, epochs=epochs_lstm, batch_size=batch_size, validation_split=0.1, callbacks = [es])

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 52ms/step - accuracy: 0.0379 - loss: 6.7614 - val_accuracy: 0.0386 - val_loss: 6.6840
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 30s 50ms/step - accuracy: 0.0561 - loss: 6.3344 - val_accuracy: 0.0645 - val_loss: 6.5799
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 51ms/step - accuracy: 0.0780 - loss: 6.0717 - val_accuracy: 0.0863 - val_loss: 6.4644
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 30s 51ms/step - accuracy: 0.0975 - loss: 5.8412 - val_accuracy: 0.0962 - val_loss: 6.4263
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 51ms/step - accuracy: 0.1103 - loss: 5.6511 - val_accuracy: 0.1024 - val_loss: 6.4352
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1181 - loss: 5.4784 - val_accuracy: 0.1044 - val_loss: 6.4405
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 51ms/step - accuracy: 0.1261 - loss: 5.3211 - val_accuracy: 0.1066 - val_loss: 6.4615
